# Coletor de Dados do YouTube via API — versão Notebook### Recuperação de Informação na Web e Mídias SociaisEste notebook é a versão Colab do coletor de dados do YouTube. Ele:- **Monta o Google Drive** e salva tudo numa pasta que você escolhe;- Busca vídeos por **query + intervalo de datas** (mês a mês), coletando **vídeos, canais e comentários** (com replies);- Faz **rotação de chaves de API** quando a quota estoura;- Mantém **estado de retomada** — se a coleta cair, ela continua de onde parou (grava a cada lote direto no Drive).> **Degrau 1 da escada de recuperação:** o YouTube oferece API oficial. Coletar por aqui é a forma estável, permitida e estruturada — não scraping.

---## 1. Montar o Google DriveA coleta grava **direto na pasta do Drive** a cada lote, para sobreviver a quedas de conexão ou de runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---## 2. Configuração (edite esta célula)Tudo que você precisa ajustar está aqui — equivale ao antigo `config.py`. A coleta ocorre **da data final para a inicial** (mais recente para mais antiga).

In [ ]:
# ====================== CONFIGURAÇÃO ======================
config = {
    # Pasta DESTINO dentro do seu Drive (será criada se não existir).
    # Tudo é salvo aqui: CSVs de saída + arquivos de estado para retomada.
    "drive_folder": "/content/drive/MyDrive/coleta_youtube",

    # Região e idioma da coleta
    "region_code": "BR",          # ISO 3166-1 alpha-2
    "relevance_language": "pt",   # ISO 639-1

    # Janela de coleta -> [ano, mês, dia]  (coleta de end_date para start_date)
    "start_date": [2025, 1, 1],
    "end_date":   [2025, 1, 23],

    # Intervalo de varredura: "monthly" ou "weekly"
    "interval_type": "monthly",

    # Filtra TÍTULOS por estas palavras. Deixe [] para não filtrar.
    "key_words": [],

    # Chaves da API v3 do YouTube (uma ou várias; a rotação usa todas)
    "youtube_keys": [
        "SUA_CHAVE_AQUI",
        # "chave_2",
        # "chave_3",
    ],

    # Buscas que serão usadas
    "queries": [
        "busca sobre o tema 1",
        "busca sobre o tema 2",
    ],

    # Tempo (s) de nova tentativa após falha de conexão/HTTP genérica
    "try_again_timeout": 60,

    # Quando a quota de TODAS as chaves estoura, quantas horas esperar
    "quota_sleep_hours": 5,
}
# =========================================================
print("Config carregada. Pasta destino:", config["drive_folder"])

---## 3. Preparação do ambienteInstala a biblioteca cliente do Google e define utilitários que, no script original, vinham de módulos separados (`log`, estado global, etc.). Aqui ficam embutidos para o notebook ser autossuficiente.

In [ ]:
!pip -q install google-api-python-client >/dev/null 2>&1

import os, csv, time, json, requests, socket
import pandas as pd
from datetime import datetime, timedelta
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Timeout de socket (3 min), como no script original
socket.setdefaulttimeout(60 * 3)

# Pasta de saída no Drive (substitui o antigo diretório local "files")
FILES = config["drive_folder"]
os.makedirs(FILES, exist_ok=True)

def fpath(name):
    """Caminho de um arquivo dentro da pasta de saída no Drive."""
    return os.path.join(FILES, name)

# --- Utilitários que no script vinham de scripts/* ---
def log(tag, msg):
    """Log simples com timestamp (substitui scripts.console.log)."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}][{tag}] {msg}")

class GlobalState:
    """Estado em memória (substitui scripts.globalState). Aqui só registra/loga."""
    _inst = None
    def __init__(self): self.state = {}
    @staticmethod
    def get_instance():
        if GlobalState._inst is None: GlobalState._inst = GlobalState()
        return GlobalState._inst
    def set_state(self, k, v): self.state[k] = v

def secondsUntil(hours):
    """Segundos a esperar (substitui scripts.secondsUntil). Simplificado: N horas."""
    return int(hours * 3600)

def connectCheckAPI():
    """No script original conectava a uma API de status externa. Aqui é no-op."""
    pass

print("Ambiente pronto. Saída em:", FILES)

---## 4. Gerenciador da API com rotação de chavesFiel ao script: quando uma chave estoura a quota (`quotaExceeded`), passa para a próxima. Se todas estouram, dorme e recomeça. Cada requisição é registrada em `requisicoes.csv` no Drive.

In [ ]:
class YouTubeAPIManager:
    YOUTUBE_API_SERVICE_NAME = 'youtube'
    YOUTUBE_API_VERSION = 'v3'
    _inst = None

    def __init__(self):
        self.current_key_index = -1
        self.DEVELOPER_KEYS = config["youtube_keys"]
        self.youtube = self.get_new_youtube_client()

    @staticmethod
    def get_instance():
        if YouTubeAPIManager._inst is None:
            YouTubeAPIManager._inst = YouTubeAPIManager()
        return YouTubeAPIManager._inst

    def get_new_youtube_client(self):
        self.DEVELOPER_KEYS = config['youtube_keys']
        if self.current_key_index >= len(self.DEVELOPER_KEYS) - 1:
            timeout = secondsUntil(config["quota_sleep_hours"])
            log("key", f"Todas as chaves excederam a quota. Aguardando {timeout}s.")
            GlobalState.get_instance().set_state("status", "sleeping")
            time.sleep(timeout)
            GlobalState.get_instance().set_state("status", "working")
            self.current_key_index = 0
        else:
            self.current_key_index += 1
        developerKey = self.DEVELOPER_KEYS[self.current_key_index]
        GlobalState.get_instance().set_state(
            "key_progress", f"{self.current_key_index + 1}/{len(self.DEVELOPER_KEYS)}")
        log("key", f"Usando chave {self.current_key_index + 1}/{len(self.DEVELOPER_KEYS)}")
        return build(self.YOUTUBE_API_SERVICE_NAME, self.YOUTUBE_API_VERSION,
                     developerKey=developerKey)

    def make_api_request(self, method_func, **kwargs):
        with open(fpath("requisicoes.csv"), mode='a', newline='') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=kwargs.keys())
            if csv_file.tell() == 0:
                writer.writeheader()
            while True:
                try:
                    writer.writerow(kwargs)
                    request = method_func(self.youtube, **kwargs)
                    return request.execute()
                except HttpError as e:
                    dados_json = json.loads(e.content) if hasattr(e, 'content') else {}
                    try:
                        razao = dados_json["error"]["errors"][0]["reason"]
                    except Exception:
                        razao = ""
                    if razao == "quotaExceeded":
                        log("key", "Quota excedida; trocando de chave...")
                        self.youtube = self.get_new_youtube_client()
                    elif razao == "commentsDisabled":
                        log("video", "Comentários desabilitados.")
                        raise HttpError(e.resp, e.content, uri=e.uri)
                    elif e.resp.status == 403:
                        log("video", "Acesso restrito (403); pulando.")
                        return None
                    else:
                        log("error", f"Erro HTTP; nova tentativa em {config['try_again_timeout']}s")
                        time.sleep(config["try_again_timeout"])
                except Exception:
                    log("error", f"Problema de conexão; nova tentativa em {config['try_again_timeout']}s")
                    time.sleep(config["try_again_timeout"])

---## 5. Funções de coletaDetalhes de **vídeo**, **canal**, **comentários** e **replies** — fiéis ao script original.

In [ ]:
def generate_date_intervals(start_date, end_date, interval_type):
    interval_delta = {"weekly": timedelta(weeks=1), "monthly": timedelta(days=30)}
    current_start = end_date
    while current_start > start_date:
        current_end = min(current_start, end_date)
        current_start = max(current_end - interval_delta[interval_type], start_date)
        yield current_start, current_end

def get_video_details(video_id):
    api = YouTubeAPIManager.get_instance()
    method = lambda client, **kw: client.videos().list(**kw)
    resp = api.make_api_request(
        method, id=video_id,
        part='snippet,statistics,contentDetails,status,liveStreamingDetails,localizations,topicDetails,recordingDetails')
    if resp is None or not resp.get('items'):
        return None
    v = resp['items'][0]
    sn, cd = v['snippet'], v['contentDetails']
    st, stat = v['status'], v['statistics']
    live = v.get('liveStreamingDetails', {})
    topic = v.get('topicDetails', {})
    rec = v.get('recordingDetails', {})
    return {
        "video_id": video_id,
        "title": sn.get('title'),
        "description": sn.get('description'),
        "channel_id": sn.get('channelId'),
        "published_at": sn.get('publishedAt'),
        "category_id": sn.get('categoryId', ""),
        "tags": sn.get('tags', []),
        "view_count": int(stat.get('viewCount', 0)),
        "like_count": int(stat.get('likeCount', 0)),
        "comment_count": int(stat.get('commentCount', 0)),
        "duration": cd.get('duration'),
        "definition": cd.get('definition'),
        "caption": cd.get('caption') == 'true',
        "licensed_content": cd.get('licensedContent', False),
        "privacy_status": st.get('privacyStatus'),
        "license": st.get('license'),
        "embeddable": st.get('embeddable', False),
        "public_stats_viewable": st.get('publicStatsViewable', False),
        "is_made_for_kids": st.get('madeForKids', False),
        "thumbnail_url": sn.get('thumbnails', {}).get('high', {}).get('url'),
        "default_audio_language": sn.get('defaultAudioLanguage'),
        "default_language": sn.get('defaultLanguage'),
        "actual_start_time": live.get('actualStartTime', ''),
        "scheduled_start_time": live.get('scheduledStartTime', ''),
        "actual_end_time": live.get('actualEndTime', ''),
        "scheduled_end_time": live.get('scheduledEndTime', ''),
        "concurrent_viewers": live.get('concurrentViewers', 0),
        "recording_date": rec.get('recordingDate', ''),
        "topicCategories": topic.get('topicCategories', []),
    }

def get_channel_details(channel_id):
    api = YouTubeAPIManager.get_instance()
    method = lambda client, **kw: api.youtube.channels().list(**kw)
    resp = api.make_api_request(
        method, part="snippet,statistics,contentDetails,brandingSettings", id=channel_id)
    if not resp or not resp.get('items'):
        return None
    c = resp['items'][0]
    sn, stat = c['snippet'], c['statistics']
    branding = c.get('brandingSettings', {})
    return {
        "channel_id": channel_id,
        "title": sn.get('title', ""),
        "description": sn.get('description', ""),
        "published_at": sn.get('publishedAt', ""),
        "country": sn.get('country', ""),
        "view_count": int(stat.get('viewCount', 0)),
        "comment_count": int(stat.get('commentCount', 0)),
        "subscriber_count": int(stat.get('subscriberCount', 0)),
        "video_count": int(stat.get('videoCount', 0)),
        "keywords": branding.get('channel', {}).get('keywords', ""),
        "profile_picture_url": sn.get('thumbnails', {}).get('default', {}).get('url', ""),
    }

In [ ]:
def get_replies(video_id, comment_id):
    replies, page_token = [], None
    api = YouTubeAPIManager.get_instance()
    while True:
        method = lambda client, **kw: api.youtube.comments().list(**kw)
        try:
            resp = api.make_api_request(
                method, part="snippet", parentId=comment_id,
                maxResults=100, pageToken=page_token, textFormat="plainText")
            for item in resp.get("items", []):
                ri = item["snippet"]
                replies.append({
                    "video_id": video_id, "comment_id": item["id"],
                    "author": ri.get("authorDisplayName"),
                    "author_channel_id": ri.get("authorChannelId", {}).get("value"),
                    "comment": ri.get("textDisplay"),
                    "published_at": ri.get("publishedAt"),
                    "updated_at": ri.get("updatedAt", ""),
                    "like_count": ri.get("likeCount"),
                    "is_reply": True, "parent_id": comment_id,
                })
            page_token = resp.get('nextPageToken')
            if not page_token:
                break
        except HttpError:
            log("error", "Erro ao coletar replies; interrompendo este ramo.")
            break
    return replies

def get_comments(video_id, total_comment_count):
    api = YouTubeAPIManager.get_instance()
    comments, page_token, collected = [], None, 0
    while True:
        try:
            method = lambda client, **kw: api.youtube.commentThreads().list(**kw)
            resp = api.make_api_request(
                method, part="snippet,replies", videoId=video_id,
                maxResults=100, pageToken=page_token, textFormat="plainText")
        except HttpError as e:
            try:
                razao = json.loads(e.content)["error"]["errors"][0]["reason"]
            except Exception:
                razao = ""
            if e.resp.status == 404 or razao == "commentsDisabled":
                log("video", f"Sem comentários acessíveis em {video_id}; pulando.")
                return comments
            raise
        for item in resp.get("items", []):
            collected += 1
            ci = item["snippet"]["topLevelComment"]["snippet"]
            cid = item["snippet"]["topLevelComment"]["id"]
            comments.append({
                "video_id": video_id, "comment_id": cid,
                "author": ci.get("authorDisplayName"),
                "author_channel_id": ci.get("authorChannelId", {}).get("value"),
                "comment": ci.get("textDisplay"),
                "published_at": ci.get("publishedAt"),
                "updated_at": ci.get("updatedAt", None),
                "like_count": ci.get("likeCount"),
                "is_reply": False, "parent_id": None,
            })
            if item["snippet"]["totalReplyCount"] > 0:
                comments.extend(get_replies(video_id, cid))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    log("comments", f"Coletados {collected} comentários de {video_id}.")
    return comments

---## 6. Processar um vídeo e salvar **direto no Drive**Cada vídeo processado grava imediatamente em `videos_info.csv`, `channels_info.csv` e `comments_info.csv` na sua pasta do Drive (modo *append*), e registra o ID em `processed_videos.csv`. Assim, se a coleta cair, nada do que já foi salvo se perde.

In [ ]:
def process_video(video_id, processed_videos, video_details=None):
    if video_details is None:
        video_details = get_video_details(video_id)
    if video_details is None:
        log("video", f"{video_id}: detalhes indisponíveis (restrição); pulando.")
        return

    total_comments = video_details['comment_count']

    # Mantém a salvaguarda do script: ignora vídeos com excesso de comentários
    if 0 < total_comments < 30000:
        videos_exists = os.path.isfile(fpath('videos_info.csv'))
        channels_exists = os.path.isfile(fpath('channels_info.csv'))
        comments_exists = os.path.isfile(fpath('comments_info.csv'))

        pd.DataFrame([video_details]).to_csv(
            fpath('videos_info.csv'), mode='a', header=not videos_exists, index=False)

        channel_details = get_channel_details(video_details['channel_id'])
        if channel_details:
            pd.DataFrame([channel_details]).to_csv(
                fpath('channels_info.csv'), mode='a', header=not channels_exists, index=False)

        comments = get_comments(video_id, total_comments)
        if comments:
            cdf = pd.DataFrame(comments)
            cdf['channel_id'] = video_details['channel_id']
            cdf.to_csv(fpath('comments_info.csv'), mode='a',
                       header=not comments_exists, index=False)

    processed_videos.add(video_id)
    with open(fpath('processed_videos.csv'), 'a', newline='') as f:
        csv.writer(f).writerow([video_id])

def make_search_request(query, published_after, published_before, region, language):
    api = YouTubeAPIManager.get_instance()
    method = lambda client, **kw: api.youtube.search().list(**kw)
    log("search", f"Nova query: {query}")
    resp = api.make_api_request(
        method, part="id,snippet", q=query, maxResults=50, type="video",
        order="relevance", publishedAfter=published_after,
        publishedBefore=published_before, regionCode=region, relevanceLanguage=language)
    n = len(resp.get('items', [])) if resp else 0
    log("search", f"A query retornou {n} vídeos.")
    return resp or {"items": []}

---## 7. Estado e retomadaEquivale ao `reset.py` + leitura de `atual_date.csv`. A coleta começa da **data atual de progresso** (que avança a cada intervalo) até a data inicial. Rode a célula de **reset** apenas quando quiser começar do zero — ela preserva a coleta anterior renomeando a pasta.

In [ ]:
# === RESET (opcional) — rode só para começar uma coleta do zero ===
# Preserva a coleta anterior renomeando a pasta e recria o estado inicial.

def reset_coleta():
    if os.path.isdir(FILES) and os.listdir(FILES):
        backup = FILES.rstrip('/') + f" {datetime.now().strftime('%Y%m%d_%H%M%S')}"
        os.rename(FILES, backup)
        log("reset", f"Pasta anterior preservada em: {backup}")
    os.makedirs(FILES, exist_ok=True)
    with open(fpath("atual_date.csv"), "w", newline="") as f:
        csv.DictWriter(f, fieldnames=["year", "month", "day"]).writerow({
            "year": config["end_date"][0],
            "month": config["end_date"][1],
            "day": config["end_date"][2]})
    log("reset", "Estado reiniciado a partir de end_date.")

# Descomente para resetar:
# reset_coleta()

In [ ]:
def carregar_data_atual():
    """Lê atual_date.csv (progresso). Se não existir, inicia em end_date."""
    p = fpath("atual_date.csv")
    if not os.path.isfile(p):
        with open(p, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=["year", "month", "day"]).writerow({
                "year": config["end_date"][0], "month": config["end_date"][1],
                "day": config["end_date"][2]})
    df = pd.read_csv(p, header=None)
    return datetime(int(df.iloc[0, 0]), int(df.iloc[0, 1]), int(df.iloc[0, 2]))

def salvar_data_atual(dt):
    with open(fpath("atual_date.csv"), "w", newline="") as f:
        csv.DictWriter(f, fieldnames=["year", "month", "day"]).writerow(
            {"year": dt.year, "month": dt.month, "day": dt.day})

def carregar_processados():
    p = fpath("processed_videos.csv")
    if os.path.isfile(p):
        with open(p, 'r') as f:
            return {row[0] for row in csv.reader(f) if row}
    return set()

---## 8. Executar a coletaRoda a coleta percorrendo intervalos (da data atual de progresso para a inicial) e, em cada um, todas as queries. Salva tudo no Drive em tempo real. Pode interromper e rodar de novo: continua de onde parou.

In [ ]:
def main():
    connectCheckAPI()
    GlobalState.get_instance().set_state("status", "working")

    start_date = datetime(*config['start_date'])
    end_date = carregar_data_atual()
    processed = carregar_processados()
    queries = config["queries"]
    region, language = config["region_code"], config["relevance_language"]
    key_words = [w.lower() for w in config["key_words"]]

    log("main", f"Coleta de {end_date.date()} até {start_date.date()} | "
                f"{len(queries)} queries | {len(processed)} vídeos já processados.")

    for start_iv, end_iv in generate_date_intervals(start_date, end_date, config["interval_type"]):
        log("interval", f"[{start_iv.date()} - {end_iv.date()}]")
        salvar_data_atual(end_iv)  # avança o progresso (permite retomada)

        for i, query in enumerate(queries, 1):
            GlobalState.get_instance().set_state("query_progress", f"{i}/{len(queries)}")
            after = start_iv.isoformat() + "Z"
            before = end_iv.isoformat() + "Z"
            resp = make_search_request(query, after, before, region, language)
            videos = resp.get("items", [])
            if not videos:
                continue

            for idx, item in enumerate(videos, 1):
                title = item['snippet']['title'].lower()
                # filtra por palavra-chave (ou aceita tudo se a lista estiver vazia)
                if not key_words or any(w in title for w in key_words):
                    video_id = item['id']['videoId']
                    if video_id in processed:
                        continue
                    log("video", f"[{idx}/{len(videos)}] {video_id}")
                    details = get_video_details(video_id)
                    if details and details['comment_count'] > 0:
                        process_video(video_id, processed, video_details=details)

            log("search", f"Concluído: '{query}' em [{start_iv.date()} - {end_iv.date()}]")

    log("main", "Coleta finalizada (ou nada mais a coletar no intervalo).")

# Para iniciar, descomente:
# main()

---## 9. Conferir o que foi coletadoLê os CSVs salvos no Drive e mostra um resumo.

In [ ]:
def resumo():
    for nome in ["videos_info.csv", "channels_info.csv", "comments_info.csv"]:
        p = fpath(nome)
        if os.path.isfile(p):
            df = pd.read_csv(p)
            print(f"{nome}: {len(df)} linhas")
        else:
            print(f"{nome}: (ainda não criado)")

resumo()

# Exemplo: ver os primeiros vídeos coletados
# pd.read_csv(fpath('videos_info.csv')).head()

---### Notas- **Chaves de API:** crie em <https://console.cloud.google.com/> (YouTube Data API v3). Quanto mais chaves em `youtube_keys`, mais quota diária somada — a rotação usa todas.- **Custo de quota:** `search` custa 100 unidades por chamada; `videos`/`channels`/`commentThreads` custam 1. Buscas largas esgotam quota rápido — restrinja `queries`, `key_words` e a janela de datas.- **Retomada:** o progresso fica em `atual_date.csv` e os IDs já vistos em `processed_videos.csv`, ambos no Drive. Reexecutar `main()` continua de onde parou.- **Ética (degrau 1):** estamos usando a API oficial e respeitando suas quotas — a forma permitida e estável de recuperar estes dados.